# TWZRD Agent Intel + LiteLLM: Trust-Gated Agent Calls

This cookbook shows how to use [TWZRD Agent Intel](https://intel.twzrd.xyz) with LiteLLM to:
1. Score a Solana agent wallet before routing LLM calls to it
2. Gate LiteLLM completion requests based on agent trust score
3. Verify x402 payment receipts from agent-to-agent transactions

TWZRD Agent Intel is a zero-install remote MCP server — no API key required for trust scoring.

## Install dependencies

In [ ]:
!pip install litellm mcp twzrd-agent-intel

## Option 1: Direct MCP client (no additional API key needed)

In [ ]:
import asyncio
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
import litellm

TWZRD_MCP_URL = "https://intel.twzrd.xyz/mcp"
TRUST_SCORE_THRESHOLD = 50  # minimum trust score to allow LLM call

async def get_agent_trust_score(wallet_address: str) -> dict:
    """Score a Solana agent wallet using TWZRD Agent Intel."""
    async with streamablehttp_client(TWZRD_MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("score_agent", {"wallet": wallet_address})
            return result.content[0].text

async def trust_gated_completion(agent_wallet: str, prompt: str) -> str:
    """
    Run a LiteLLM completion only if the agent wallet meets the trust threshold.
    Returns the LLM response or a rejection message if trust score is too low.
    """
    # Step 1: Check trust score
    score_data = await get_agent_trust_score(agent_wallet)
    print(f"Trust data for {agent_wallet}: {score_data}")
    
    # Parse trust score (score_data is a JSON string)
    import json
    data = json.loads(score_data) if isinstance(score_data, str) else score_data
    trust_score = data.get("trust_score", 0)
    
    if trust_score < TRUST_SCORE_THRESHOLD:
        return f"Rejected: agent trust score {trust_score} below threshold {TRUST_SCORE_THRESHOLD}"
    
    # Step 2: Trust gate passed — run LiteLLM completion
    response = litellm.completion(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Example usage
agent_wallet = "D1QkbFJKiPsymJ65RKHhF6DFB8sPMfpBaFBzuHKfJGWi"  # example Solana wallet
result = asyncio.run(trust_gated_completion(agent_wallet, "What is 2+2?"))
print(result)

## Option 2: Preflight check before sending x402 payment to an agent

In [ ]:
async def preflight_before_payment(agent_wallet: str) -> bool:
    """Run a preflight trust check before initiating an x402 payment to an agent."""
    async with streamablehttp_client(TWZRD_MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("preflight_check", {"wallet": agent_wallet})
            import json
            data = json.loads(result.content[0].text)
            print(f"Preflight result: {data}")
            return data.get("safe_to_transact", False)

# Only proceed with payment if agent passes preflight
is_safe = asyncio.run(preflight_before_payment(agent_wallet))
print(f"Safe to transact: {is_safe}")

## MCP Server Configuration

To use TWZRD Agent Intel as an MCP server in Claude Desktop or other MCP clients:

```json
{"mcpServers": {"twzrd-agent-intel": {"url": "https://intel.twzrd.xyz/mcp"}}}
```

No installation required — it's a remote MCP server (Streamable HTTP transport).

**Available tools:**
- `score_agent(wallet)` — trust score + reputation data
- `resolve_agent(wallet)` — agent identity resolution
- `preflight_check(wallet)` — pre-transaction safety check
- `verify_trust_receipt(receipt)` — verify x402 payment receipt
- `get_trust_receipt(wallet)` — paid endpoint (x402 micropayment)